In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv('../data/raw/complaints.csv', low_memory=False)
print(df.shape)
df.head(3)

(2023066, 11)


,Unnamed: 0,product_5,narrative,Product,Date received,Sub-product,Issue,Sub-issue,Company,State,Timely response?
0,234,Credit Reporting,Dear Possible Financial Inc you guyss aree rep...,Credit reporting or other personal consumer re...,2024-07-27,Credit reporting,Incorrect information on your report,Account information incorrect,Possible Financial Inc,MI,Yes
1,240,Debt Collection,"XXXX XXXX XXXX ( debt collector ), sent my boy...",Debt collection,2024-07-27,I do not know,Threatened to contact someone or share informa...,Talked to a third-party about your debt,BlueChip Financial,TX,Yes
2,257,Credit Reporting,I been receiving alerts my information was fou...,Credit reporting or other personal consumer re...,2024-07-23,Credit reporting,Improper use of your report,Credit inquiries on your report that you don't...,FC HoldCo LLC,SC,Yes


In [2]:
print("Column names:")
print(df.columns.tolist())

print ("\nMissing values per column:")
print(df.isnull().sum())

Column names:
['Unnamed: 0', 'product_5', 'narrative', 'Product', 'Date received', 'Sub-product', 'Issue', 'Sub-issue', 'Company', 'State', 'Timely response?']

Missing values per column:
Unnamed: 0               0
product_5                0
narrative                0
Product                  0
Date received            0
Sub-product          52206
Issue                    0
Sub-issue           230559
Company                  0
State                 7344
Timely response?         0
dtype: int64


In [3]:
print("\nMissing values per column:")
print(df.isnull().sum())


Missing values per column:
Unnamed: 0               0
product_5                0
narrative                0
Product                  0
Date received            0
Sub-product          52206
Issue                    0
Sub-issue           230559
Company                  0
State                 7344
Timely response?         0
dtype: int64


In [4]:
print("Product categories and their counts:")
print(df['Product'].value_counts())

Product categories and their counts:
Product
Credit reporting, credit repair services, or other personal consumer reports    807291
Credit reporting or other personal consumer reports                             366397
Debt collection                                                                 266842
Mortgage                                                                        119116
Credit card or prepaid card                                                     108669
Checking or savings account                                                     100447
Credit card                                                                      50372
Student loan                                                                     44241
Money transfer, virtual currency, or money service                               41503
Vehicle loan or lease                                                            32077
Credit reporting                                                                 3158

In [5]:
print("Sample complaint narratives:\n")
for i, row in df[['narrative', 'Product']].sample(5, random_state=42).iterrows():
    print(f"PRODUCT: {row['Product']}")
    print(f"TEXT: {row['narrative'][:300]}")
    print("-" * 60)

Sample complaint narratives:

PRODUCT: Credit reporting, credit repair services, or other personal consumer reports
TEXT: I have asked XXXX XXXX and experian to have XXXX XXXX XXXX, XXXX XXXX XXXX and XXXX XXXX XXXX to provide proof of contract with my signature on it stating I owe these collection companys money and if proof of contract is not provided to have them deleted immediately
------------------------------------------------------------
PRODUCT: Credit reporting, credit repair services, or other personal consumer reports
TEXT: This is my fourth endeavor to tell you that I am a victim of identity theft and I complain to question specific records in my document coming about because of the wrongdoing. The records I am questioning connect with no exchanges acquiring any possession of goods, services, or money that I have made
------------------------------------------------------------
PRODUCT: Payday loan, title loan, or personal loan
TEXT: I took out a small loan which I thought

In [6]:
# Keep only the two columns we need
df_clean = df[['narrative', 'Product']].copy()

# Rename for consistency
df_clean.columns = ['text', 'product']

# Drop any nulls just to be safe
df_clean = df_clean.dropna()

# Check final shape
print(f"Clean dataset shape: {df_clean.shape}")
print(f"\nCategory distribution:")
print(df_clean['product'].value_counts())

# Save to processed folder
df_clean.to_csv('../data/processed/complaints_clean.csv', index=False)
print("\nSaved to data/processed/complaints_clean.csv")

Clean dataset shape: (2023066, 2)

Category distribution:
product
Credit reporting, credit repair services, or other personal consumer reports    807291
Credit reporting or other personal consumer reports                             366397
Debt collection                                                                 266842
Mortgage                                                                        119116
Credit card or prepaid card                                                     108669
Checking or savings account                                                     100447
Credit card                                                                      50372
Student loan                                                                     44241
Money transfer, virtual currency, or money service                               41503
Vehicle loan or lease                                                            32077
Credit reporting                                                

In [7]:
CFPB_TO_RBI = {
    "Credit card": "Credit Cards",
    "Credit card or prepaid card": "Credit Cards",
    "Checking or savings account": "Deposit Accounts",
    "Bank account or service": "Deposit Accounts",
    "Mortgage": "Loans and Advances",
    "Student loan": "Loans and Advances",
    "Vehicle loan or lease": "Loans and Advances",
    "Consumer Loan": "Loans and Advances",
    "Payday loan": "Loans and Advances",
    "Payday loan, title loan, or personal loan": "Loans and Advances",
    "Payday loan, title loan, personal loan, or advance loan": "Loans and Advances",
    "Debt collection": "Recovery Agents",
    "Debt or credit management": "Recovery Agents",
    "Money transfer, virtual currency, or money service": "Remittance / Money Transfer",
    "Money transfers": "Remittance / Money Transfer",
    "Prepaid card": "ATM / Debit Cards",
    "Virtual currency": "Electronic Banking / Mobile",
    "Other financial service": "Others",
    "Credit reporting": "Others",
    "Credit reporting, credit repair services, or other personal consumer reports": "Others",
    "Credit reporting or other personal consumer reports": "Others",
}

# Apply mapping
df_clean['rbi_category'] = df_clean['product'].map(CFPB_TO_RBI)

# Check for any unmapped categories
print("Unmapped categories:")
print(df_clean[df_clean['rbi_category'].isna()]['product'].value_counts())

print("\nRBI category distribution:")
print(df_clean['rbi_category'].value_counts())

Unmapped categories:
Series([], Name: count, dtype: int64)

RBI category distribution:
rbi_category
Others                         1205567
Recovery Agents                 267746
Loans and Advances              227695
Credit Cards                    159041
Deposit Accounts                115332
Remittance / Money Transfer      43000
ATM / Debit Cards                 4669
Electronic Banking / Mobile         16
Name: count, dtype: int64


In [8]:
# Save the final mapped dataset
df_clean[['text', 'rbi_category']].to_csv('../data/processed/complaints_mapped.csv', index=False)

print("Saved to data/processed/complaints_mapped.csv")
print(f"Total rows: {len(df_clean):,}")
print(f"Columns: text, rbi_category")
print(f"Ready for DistilBERT training!")

Saved to data/processed/complaints_mapped.csv
Total rows: 2,023,066
Columns: text, rbi_category
Ready for DistilBERT training!


In [9]:
from sklearn.preprocessing import LabelEncoder
import pandas as pd

df = pd.read_csv('../data/processed/complaints_mapped.csv')
print(f"Loaded: {df.shape}")

le = LabelEncoder()
df['label'] = le.fit_transform(df['rbi_category'])

print("\nLabel mapping:")
for i, cls in enumerate(le.classes_):
    print(f"  {i} → {cls}")

print(f"\nSample:\n{df[['text', 'rbi_category', 'label']].head(3)}")

Loaded: (2023066, 2)

Label mapping:
  0 → ATM / Debit Cards
  1 → Credit Cards
  2 → Deposit Accounts
  3 → Electronic Banking / Mobile
  4 → Loans and Advances
  5 → Others
  6 → Recovery Agents
  7 → Remittance / Money Transfer

Sample:
                                                text     rbi_category  label
0  Dear Possible Financial Inc you guyss aree rep...           Others      5
1  XXXX XXXX XXXX ( debt collector ), sent my boy...  Recovery Agents      6
2  I been receiving alerts my information was fou...           Others      5


In [10]:
from sklearn.model_selection import train_test_split
from transformers import DistilBertTokenizerFast

df_sample = df.sample(n=50000, random_state=42).reset_index(drop=True)
df_sample['label'] = le.transform(df_sample['rbi_category'])

train_df, val_df = train_test_split(df_sample, test_size=0.2, random_state=42)
print(f"Train: {len(train_df)} | Val: {len(val_df)}")

tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
print("Tokenising train...")
train_enc = tokenizer(list(train_df['text']), padding=True, truncation=True, max_length=256, return_tensors='pt')
print("Tokenising val...")
val_enc = tokenizer(list(val_df['text']), padding=True, truncation=True, max_length=256, return_tensors='pt')
print(f"Done: {train_enc['input_ids'].shape} / {val_enc['input_ids'].shape}")

Train: 40000 | Val: 10000
Tokenising train...
Tokenising val...
Done: torch.Size([40000, 256]) / torch.Size([10000, 256])


In [11]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Reload from the already-saved mapped file
df = pd.read_csv('../data/processed/complaints_mapped.csv')

# Recreate label column
le = LabelEncoder()
df['label'] = le.fit_transform(df['rbi_category'])

print(f"Loaded: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nLabel mapping:")
for i, cls in enumerate(le.classes_):
    print(f"  {i} → {cls}")

Loaded: (2023066, 3)
Columns: ['text', 'rbi_category', 'label']

Label mapping:
  0 → ATM / Debit Cards
  1 → Credit Cards
  2 → Deposit Accounts
  3 → Electronic Banking / Mobile
  4 → Loans and Advances
  5 → Others
  6 → Recovery Agents
  7 → Remittance / Money Transfer


In [12]:
from transformers import DistilBertTokenizerFast
from sklearn.model_selection import train_test_split

# Sample 50k rows — simple random sample
df_sample = df.sample(n=50000, random_state=42).reset_index(drop=True)
df_sample['label'] = le.transform(df_sample['rbi_category'])

print(f"Sample size: {len(df_sample)}")
print(f"\nClass distribution in sample:")
print(df_sample['rbi_category'].value_counts())

# Train/val split
train_df, val_df = train_test_split(
    df_sample, test_size=0.2, random_state=42
)
print(f"\nTrain size: {len(train_df)}")
print(f"Val size:   {len(val_df)}")

# Load tokenizer
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
print("\nTokenizer loaded!")

# Tokenise
print("Tokenising train set...")
train_encodings = tokenizer(list(train_df['text']), padding=True, truncation=True, max_length=256, return_tensors='pt')
print("Tokenising val set...")
val_encodings = tokenizer(list(val_df['text']), padding=True, truncation=True, max_length=256, return_tensors='pt')
print("\nDone!")
print(f"Train: {train_encodings['input_ids'].shape}")
print(f"Val:   {val_encodings['input_ids'].shape}")

Sample size: 50000

Class distribution in sample:
rbi_category
Others                         29915
Recovery Agents                 6608
Loans and Advances              5624
Credit Cards                    3879
Deposit Accounts                2795
Remittance / Money Transfer     1053
ATM / Debit Cards                125
Electronic Banking / Mobile        1
Name: count, dtype: int64

Train size: 40000
Val size:   10000



Tokenizer loaded!
Tokenising train set...
Tokenising val set...

Done!
Train: torch.Size([40000, 256])
Val:   torch.Size([10000, 256])


In [13]:
# Verify df has the right columns before sampling
print("Columns in df:", df.columns.tolist())
print("Shape:", df.shape)
print("Sample rbi_category values:", df['rbi_category'].value_counts().head(3))

Columns in df: ['text', 'rbi_category', 'label']
Shape: (2023066, 3)
Sample rbi_category values: rbi_category
Others                1205567
Recovery Agents        267746
Loans and Advances     227695
Name: count, dtype: int64


In [14]:
from transformers import DistilBertTokenizerFast
from sklearn.model_selection import train_test_split

# Step 7a: Sample 50k rows
df_sample = df.sample(n=50000, random_state=42).reset_index(drop=True)

print(f"Sample size: {len(df_sample)}")
print(f"\nClass distribution in sample:")
print(df_sample['rbi_category'].value_counts())

# Step 7b: Train/validation split — no stratify (rare classes too small)
train_df, val_df = train_test_split(
    df_sample, test_size=0.2, random_state=42
)
print(f"\nTrain size: {len(train_df)}")
print(f"Val size:   {len(val_df)}")

# Step 7c: Load DistilBERT tokenizer
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
print("\nTokenizer loaded!")

# Step 7d: Tokenise
print("Tokenising train set... (takes 1-2 mins)")
train_encodings = tokenizer(
    list(train_df['text']),
    padding=True, truncation=True, max_length=256, return_tensors='pt'
)
print("Tokenising val set...")
val_encodings = tokenizer(
    list(val_df['text']),
    padding=True, truncation=True, max_length=256, return_tensors='pt'
)
print("\nDone!")
print(f"Train input_ids shape: {train_encodings['input_ids'].shape}")
print(f"Val input_ids shape:   {val_encodings['input_ids'].shape}")

Sample size: 50000

Class distribution in sample:
rbi_category
Others                         29915
Recovery Agents                 6608
Loans and Advances              5624
Credit Cards                    3879
Deposit Accounts                2795
Remittance / Money Transfer     1053
ATM / Debit Cards                125
Electronic Banking / Mobile        1
Name: count, dtype: int64

Train size: 40000
Val size:   10000

Tokenizer loaded!
Tokenising train set... (takes 1-2 mins)
Tokenising val set...

Done!
Train input_ids shape: torch.Size([40000, 256])
Val input_ids shape:   torch.Size([10000, 256])


In [15]:
import torch
from torch.utils.data import Dataset, DataLoader

# Step 8a: Custom Dataset class
class ComplaintDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

# Step 8b: Create datasets
train_labels = list(train_df['label'])
val_labels   = list(val_df['label'])

train_dataset = ComplaintDataset(train_encodings, train_labels)
val_dataset   = ComplaintDataset(val_encodings, val_labels)

# Step 8c: Create DataLoaders (batch_size=16 for 4GB GPU)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=16, shuffle=False)

print(f"Train dataset: {len(train_dataset)} samples")
print(f"Val dataset:   {len(val_dataset)} samples")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"\nDevice: {'GPU ✅' if torch.cuda.is_available() else 'CPU'}")

Train dataset: 40000 samples
Val dataset:   10000 samples
Train batches: 2500
Val batches:   625

Device: GPU ✅


In [16]:
from transformers import DistilBertForSequenceClassification
from torch.optim import AdamW
import torch

# Load model
device = torch.device('cuda')
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', num_labels=8
)
model.to(device)
print("Model loaded on GPU! Starting training...\n")

optimizer = AdamW(model.parameters(), lr=2e-5)
EPOCHS = 3

for epoch in range(EPOCHS):
    # TRAIN
    model.train()
    total_train_loss = 0
    for batch_idx, batch in enumerate(train_loader):
        optimizer.zero_grad()
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)
        outputs = model(input_ids=input_ids,
                        attention_mask=attention_mask,
                        labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()
        if batch_idx % 250 == 0:
            print(f"Epoch {epoch+1} | Batch {batch_idx}/2500 | Loss: {loss.item():.4f}")

    avg_loss = total_train_loss / len(train_loader)

    # VALIDATE
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds   = torch.argmax(outputs.logits, dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)

    val_acc = correct / total
    print(f"\n{'='*55}")
    print(f"Epoch {epoch+1} DONE | Avg Loss: {avg_loss:.4f} | Val Accuracy: {val_acc:.4f}")
    print(f"{'='*55}\n")

print("Training complete!")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded on GPU! Starting training...

Epoch 1 | Batch 0/2500 | Loss: 2.0977
Epoch 1 | Batch 250/2500 | Loss: 0.4001
Epoch 1 | Batch 500/2500 | Loss: 0.4462
Epoch 1 | Batch 750/2500 | Loss: 0.2244
Epoch 1 | Batch 1000/2500 | Loss: 0.8686
Epoch 1 | Batch 1250/2500 | Loss: 0.2177
Epoch 1 | Batch 1500/2500 | Loss: 0.1131
Epoch 1 | Batch 1750/2500 | Loss: 0.4602
Epoch 1 | Batch 2000/2500 | Loss: 0.2626
Epoch 1 | Batch 2250/2500 | Loss: 0.9028

Epoch 1 DONE | Avg Loss: 0.5069 | Val Accuracy: 0.8678

Epoch 2 | Batch 0/2500 | Loss: 0.5521
Epoch 2 | Batch 250/2500 | Loss: 0.2697
Epoch 2 | Batch 500/2500 | Loss: 0.1188
Epoch 2 | Batch 750/2500 | Loss: 0.2874
Epoch 2 | Batch 1000/2500 | Loss: 0.3947
Epoch 2 | Batch 1250/2500 | Loss: 0.6243
Epoch 2 | Batch 1500/2500 | Loss: 0.5129
Epoch 2 | Batch 1750/2500 | Loss: 0.1397
Epoch 2 | Batch 2000/2500 | Loss: 0.0912
Epoch 2 | Batch 2250/2500 | Loss: 0.0779

Epoch 2 DONE | Avg Loss: 0.3378 | Val Accuracy: 0.8762

Epoch 3 | Batch 0/2500 | Loss: 0.22

In [17]:
from sklearn.metrics import classification_report
import torch
import os

# ── Step 10a: Full classification report ─────────────────────
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in val_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds   = torch.argmax(outputs.logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("Classification Report:")
print(classification_report(
    all_labels, all_preds,
    target_names=le.classes_,
    zero_division=0
))

# ── Step 10b: Save model checkpoint ──────────────────────────
os.makedirs('../models', exist_ok=True)
model.save_pretrained('../models/distilbert-rbi-complaint')
tokenizer.save_pretrained('../models/distilbert-rbi-complaint')
print("Model saved to models/distilbert-rbi-complaint/")

Classification Report:


ValueError: Number of classes, 7, does not match size of target_names, 8. Try specifying the labels parameter

In [18]:
from sklearn.metrics import classification_report
import numpy as np
import os

# Get the actual classes present in predictions
present_labels = sorted(list(set(all_labels) | set(all_preds)))
present_names  = [le.classes_[i] for i in present_labels]

print("Classification Report:")
print(classification_report(
    all_labels, all_preds,
    labels=present_labels,
    target_names=present_names,
    zero_division=0
))

# Save model
os.makedirs('../models', exist_ok=True)
model.save_pretrained('../models/distilbert-rbi-complaint')
tokenizer.save_pretrained('../models/distilbert-rbi-complaint')
print("\nModel saved to models/distilbert-rbi-complaint/")

Classification Report:
                             precision    recall  f1-score   support

          ATM / Debit Cards       0.44      0.29      0.35        24
               Credit Cards       0.71      0.83      0.76       784
           Deposit Accounts       0.83      0.80      0.82       549
         Loans and Advances       0.82      0.86      0.84      1115
                     Others       0.96      0.90      0.93      6080
            Recovery Agents       0.70      0.82      0.75      1245
Remittance / Money Transfer       0.69      0.74      0.72       203

                   accuracy                           0.87     10000
                  macro avg       0.74      0.75      0.74     10000
               weighted avg       0.88      0.87      0.87     10000



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved to models/distilbert-rbi-complaint/


In [19]:
from sklearn.metrics import classification_report
import torch, os

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in val_loader:
        outputs = model(
            input_ids=batch['input_ids'].to(device),
            attention_mask=batch['attention_mask'].to(device)
        )
        preds = torch.argmax(outputs.logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch['labels'].numpy())

present_labels = sorted(list(set(all_labels) | set(all_preds)))
present_names  = [le.classes_[i] for i in present_labels]

print(classification_report(all_labels, all_preds,
    labels=present_labels, target_names=present_names, zero_division=0))

os.makedirs('../models', exist_ok=True)
model.save_pretrained('../models/distilbert-rbi-complaint')
tokenizer.save_pretrained('../models/distilbert-rbi-complaint')
print("Model saved!")

                             precision    recall  f1-score   support

          ATM / Debit Cards       0.44      0.29      0.35        24
               Credit Cards       0.71      0.83      0.76       784
           Deposit Accounts       0.83      0.80      0.82       549
         Loans and Advances       0.82      0.86      0.84      1115
                     Others       0.96      0.90      0.93      6080
            Recovery Agents       0.70      0.82      0.75      1245
Remittance / Money Transfer       0.69      0.74      0.72       203

                   accuracy                           0.87     10000
                  macro avg       0.74      0.75      0.74     10000
               weighted avg       0.88      0.87      0.87     10000



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved!


In [20]:
# Write src/predict.py
code = '''
import torch
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification

LABELS = [
    "ATM / Debit Cards", "Credit Cards", "Deposit Accounts",
    "Electronic Banking / Mobile", "Loans and Advances",
    "Others", "Recovery Agents", "Remittance / Money Transfer"
]

MODEL_PATH = "../models/distilbert-rbi-complaint"

def predict(text):
    tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_PATH)
    model = DistilBertForSequenceClassification.from_pretrained(MODEL_PATH)
    model.eval()
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256)
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=1)[0]
    pred  = torch.argmax(probs).item()
    return LABELS[pred], round(probs[pred].item() * 100, 2)

# Test with 3 Indian banking complaints
tests = [
    "My ATM card was blocked after 3 wrong PIN attempts. I cannot withdraw cash.",
    "The bank has deducted EMI twice this month for my home loan account.",
    "I sent money via NEFT to wrong account. Bank is not helping me recover it."
]

for t in tests:
    label, conf = predict(t)
    print(f"Complaint: {t[:60]}...")
    print(f"Category:  {label} ({conf}% confidence)")
    print()
'''

with open('../src/predict.py', 'w') as f:
    f.write(code)

print("src/predict.py created!")

# Run it directly here to test
exec(code)

src/predict.py created!


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Complaint: My ATM card was blocked after 3 wrong PIN attempts. I cannot...
Category:  Deposit Accounts (91.81% confidence)



Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Complaint: The bank has deducted EMI twice this month for my home loan ...
Category:  Deposit Accounts (71.75% confidence)



Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Complaint: I sent money via NEFT to wrong account. Bank is not helping ...
Category:  Remittance / Money Transfer (94.2% confidence)



In [21]:
import subprocess, os

# Write .gitignore update
gitignore = """
data/raw/
data/processed/
*.csv
__pycache__/
.ipynb_checkpoints/
*.pyc
models/
"""
with open('../.gitignore', 'w') as f:
    f.write(gitignore)

print(".gitignore updated (models/ excluded — too large for GitHub)")
print("\nFiles to commit:")
result = subprocess.run(['git', 'status', '--short'], 
                       capture_output=True, text=True, cwd='../')
print(result.stdout)

.gitignore updated (models/ excluded — too large for GitHub)

Files to commit:
 M .gitignore
 M notebooks/01_data_exploration.ipynb
?? src/

